In [1]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from qiskit import QuantumCircuit, execute, Aer
from qiskit.providers.aer import QasmSimulator
import numpy as np
import struct

# Global variables to store the ciphertext and encryption key
global ciphertext_bits, encryption_key
ciphertext_bits = ""
encryption_key = ""

def alice_state_prep(state, basis):
    """
    Prepare Alice's quantum state based on the given state and basis choices.
    :param state: List representing Alice's qubit states (0 or 1).
    :param basis: List representing Alice's basis choices (0 for computational, 1 for Hadamard).
    :return: Quantum circuit representing the prepared state.
    """
    num_qubits = len(state)
    circuit = QuantumCircuit(num_qubits)
    for i in range(len(basis)):
        if state[i] == 1:
            circuit.x(i)  # Apply X gate if state is 1
        if basis[i] == 1:
            circuit.h(i)  # Apply Hadamard gate if basis is 1
    return circuit

def bob_measurement(circuit, basis):
    """
    Apply Bob's measurement operations based on his chosen basis.
    :param circuit: Quantum circuit prepared by Alice.
    :param basis: List representing Bob's basis choices (0 for computational, 1 for Hadamard).
    """
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)  # Apply Hadamard gate if basis is 1
    circuit.measure_all()  # Perform measurement on all qubits

def key_creation(circuit, alice_basis, bob_basis):
    """
    Generate an encryption key by comparing Alice's and Bob's basis choices.
    :param circuit: Quantum circuit with measured qubits.
    :param alice_basis: Alice's basis choices.
    :param bob_basis: Bob's basis choices.
    :return: Encryption key as a binary string.
    """
    key = execute(circuit.reverse_bits(), backend=QasmSimulator(), shots=1).result().get_counts().most_frequent()
    encryption_key = ''
    for i in range(len(alice_basis)):
        if alice_basis[i] == bob_basis[i]:  # Use only matching basis choices
            encryption_key += str(key[i])
    return encryption_key

def point_to_bits(triplet):
    """
    Convert a triplet of floating-point numbers into a binary string.
    :param triplet: Tuple of three floating-point numbers.
    :return: Binary string representation of the triplet.
    """
    return ''.join(format(struct.unpack('!Q', struct.pack('!d', t))[0], '064b') for t in triplet)

def encrypt_message(_):
    """
    Encrypt a user-provided triplet of numbers using quantum key distribution.
    """
    global ciphertext_bits, encryption_key
    try:
        triplet = tuple(map(float, triplet_input.value.split(',')))  # Parse input triplet
    except ValueError:
        with output:
            clear_output()
            print("Invalid input. Please enter three comma-separated numbers.")
        return
    
    plain_text_bits = point_to_bits(triplet)  # Convert triplet to binary
    num_qubits = int(2.0 * len(plain_text_bits))  # Set number of qubits
    
    # Alice prepares her quantum states
    alice_basis = np.random.randint(2, size=num_qubits)
    alice_state = np.random.randint(2, size=num_qubits)
    cir = alice_state_prep(alice_state, alice_basis)
    
    # Bob chooses his measurement bases and measures the qubits
    bob_basis = np.random.randint(2, size=num_qubits)
    bob_measurement(cir, bob_basis)
    
    # Generate encryption key
    encryption_key = key_creation(cir, alice_basis, bob_basis)
    encryption_key = encryption_key[:len(plain_text_bits)]  # Trim key length to match message
    
    # Encrypt the message using XOR operation
    ciphertext_bits = ''.join(str(int(plain_text_bits[i]) ^ int(encryption_key[i])) for i in range(len(plain_text_bits)))
    
    # Display the encryption results
    clear_output()
    display(triplet_input, encrypt_button, decrypt_button, output)
    with output:
        print("Ciphertext (in binary):", ciphertext_bits)
        print("Key:", encryption_key)

def binary_to_float(b):
    """
    Convert a 64-bit binary string back into a floating-point number.
    :param b: 64-bit binary string.
    :return: Floating-point number.
    """
    int_value = int(b, 2)
    return struct.unpack('!d', struct.pack('!Q', int_value))[0]

def decrypt_message(_):
    """
    Decrypt the previously encrypted message using the stored encryption key.
    """
    global ciphertext_bits, encryption_key
    if not ciphertext_bits or not encryption_key:
        with output:
            clear_output()
            print("No encrypted message found. Please encrypt a message first.")
        return
    
    # Decrypt message using XOR
    decrypted_bits = ''.join(str(int(ciphertext_bits[i]) ^ int(encryption_key[i])) for i in range(len(ciphertext_bits)))
    
    # Convert decrypted bits back into original triplet
    triplet_bits = [decrypted_bits[i:i + 64] for i in range(0, len(decrypted_bits), 64)]
    original_triplet = tuple(binary_to_float(bit) for bit in triplet_bits)
    
    # Display the decrypted triplet
    clear_output()
    display(triplet_input, encrypt_button, decrypt_button, output)
    with output:
        print("Decrypted triplet:", original_triplet)

# Create UI elements for user interaction
triplet_input = widgets.Text(description="Enter triplet:", placeholder="e.g., 35.7602,-78.188,4000")
encrypt_button = widgets.Button(description="Encrypt Message")
decrypt_button = widgets.Button(description="Decrypt Message")
encrypt_button.on_click(encrypt_message)
decrypt_button.on_click(decrypt_message)
output = widgets.Output()

# Display UI elements
display(triplet_input, encrypt_button, decrypt_button, output)


Text(value='', description='Enter triplet:', placeholder='e.g., 35.7602,-78.188,4000')

Button(description='Encrypt Message', style=ButtonStyle())

Button(description='Decrypt Message', style=ButtonStyle())

Output()